# 🌸 Clara LoRA Fine-tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chrishartline/Lily/blob/main/notebooks/clara_lora_training.ipynb)

This notebook fine-tunes TinyLlama with LoRA to give Clara her unique personality.

**Requirements:**
- GPU runtime (T4 or better)
- ~10 minutes training time

In [ ]:
# Install dependencies
%pip install -q torch transformers accelerate bitsandbytes peft trl datasets

# Check GPU
!nvidia-smi

In [ ]:
# Configuration
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./clara_lora_adapter"
MAX_SEQ_LENGTH = 512

# LoRA Configuration
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training Configuration
NUM_EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4

In [ ]:
# Load training data - Option 1: Clone from GitHub
!git clone https://github.com/chrishartline/Lily.git --depth 1 2>/dev/null || echo "Already cloned"
!cp Lily/backend/clara_training_data.jsonl . 2>/dev/null || echo "Using existing file"

# Load dataset
import json
from datasets import Dataset

examples = []
with open("clara_training_data.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            examples.append(json.loads(line.strip()))

dataset = Dataset.from_list(examples)
print(f"✅ Loaded {len(examples)} training examples")

In [ ]:
# Load model with 4-bit quantization
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"GPU: {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Train!
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE, warmup_ratio=0.03, logging_steps=10,
    fp16=True, optim="paged_adamw_8bit", report_to="none",
)

trainer = SFTTrainer(
    model=model, train_dataset=dataset, args=training_args,
    tokenizer=tokenizer, dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH,
)

print("🚀 Training...")
trainer.train()

# Save
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Saved to {OUTPUT_DIR}")

In [ ]:
# Test the fine-tuned model
def chat(message):
    system = "You are Clara (nickname Lily), the user's romantic partner and assistant."
    prompt = f"<|system|>{system}</s><|user|>{message}</s><|assistant|>"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).split('\n')[0]

for msg in ["Good morning!", "How are you?", "I love you"]:
    print(f"User: {msg}\nClara: {chat(msg)}\n")

In [ ]:
# Download the adapter
!zip -r clara_lora_adapter.zip {OUTPUT_DIR}

from google.colab import files
files.download('clara_lora_adapter.zip')
print("✅ Download started!")